# Peak Floating Point Benchmark
The `calculate_fma` function performs a number of Fused-Multiply-Add operations with 2 operation each.
Per element iteration there are 3 value reads and 1 writes and 14 arithmetic operations.
For Float64 values this results in 32 Bytes transferred.
On the AMD EPYC 9655 14 arithmetic operations per 32 Bytes is enough to reach peak performance if only the Level 1 cache is used.

You can try different data types and numbers of element per array and see how the performance changes.

In [ ]:
function calculate_fma!(a,b,c,num_elements,num_repeats)
    @inbounds @simd for j in 1:num_repeats
         for i in 1:num_elements
            x = fma(a[i],b[i],c[i]) # 2 FLOP
            y = fma(c[i],b[i],a[i]) # 2 FLOP
            z = fma(a[i],c[i],b[i]) # 2 FLOP
            k = fma(   x,   y,   z) # 2 FLOP
            l = fma(   x,   z,   y) # 2 FLOP
            m = fma(   y,   z,   x) # 2 FLOP
            n = fma(   k,   l,   m) # 2 FLOP
            c[i] = n
        end
    end
end

function measure_flops(T=Float64,num_elements=1024,num_repeats=10000000)
    num_load_store = 4    # number of load and store operations
    flop_gf        = 1e9  # FLOP per GFLOP
    num_flop       = 14   # number of floating point operations per element iteration
    bytes_GB       = 1e9     # Bytes per GB

    c = rand(T, num_elements)
    b = rand(T, num_elements)
    a = rand(T, num_elements)
    calculate_fma!(a, b, c, 1, 1) # for compilation

    e = @elapsed calculate_fma!(a, b, c, num_elements, num_repeats)

    datasize = sizeof(T) * num_elements * 3
    flops    = num_elements * num_repeats * num_flop / (flop_gf * e)
    bw       = sizeof(T) * num_elements * num_load_store * num_repeats / (bytes_GB * e)
    println("walltime     = $e sec\ndataset size = $datasize Bytes\nbandwidth    = $bw GB/s\nperformance   = $flops GFLOPS/s")
end

In [ ]:
measure_flops()

In [ ]:
measure_flops(Float32)